In [1]:
# ============================================================
# IMDb MOVIE REVIEW DATA COLLECTION AND PREPROCESSING
# PPS 1 PROJECT
# ============================================================

# Required libraries:
# pip install requests beautifulsoup4 pandas openpyxl

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re


# ============================================================
# 1. SETTINGS
# ============================================================

# IMDb Top 100 movies page
TOP_MOVIES_URL = "https://www.imdb.com/chart/top/"

# Number of movies to collect
NUMBER_OF_MOVIES = 100

# Number of reviews to collect per movie
REVIEWS_PER_MOVIE = 5

# Output files
RAW_FILE = "imdb_raw_dataset.csv"
CLEAN_FILE = "imdb_cleaned_dataset.csv"


# ============================================================
# 2. REQUEST HEADERS
# ============================================================

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/154.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}


# ============================================================
# 3. FUNCTION TO CLEAN TEXT
# ============================================================

def clean_text(text):

    if text is None:
        return ""

    # Remove HTML tags if present
    text = BeautifulSoup(str(text), "html.parser").get_text(" ")

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove unwanted characters at beginning/end
    text = text.strip()

    return text


# ============================================================
# 4. COLLECT TOP MOVIES
# ============================================================

print("=" * 60)
print("IMDb MOVIE REVIEW DATA COLLECTION")
print("=" * 60)

print("\nConnecting to IMDb...")

response = requests.get(
    TOP_MOVIES_URL,
    headers=headers,
    timeout=20
)

print("Status Code:", response.status_code)

if response.status_code != 200:
    print("\nUnable to access the IMDb page.")
    print("Do not bypass any CAPTCHA or access restriction.")
    raise SystemExit


soup = BeautifulSoup(response.text, "html.parser")


# ============================================================
# 5. FIND MOVIE INFORMATION
# ============================================================

movies = []

# IMDb current Top 250 chart uses li elements with
# data-testid="chart-layout-main-column"
movie_items = soup.select(
    'li[data-testid="chart-layout-main-column"]'
)

# Alternative selector
if not movie_items:
    movie_items = soup.select(
        'li.ipc-metadata-list-summary-item'
    )


print("\nMovies found:", len(movie_items))


for item in movie_items[:NUMBER_OF_MOVIES]:

    try:

        # ----------------------------------------------------
        # Movie name
        # ----------------------------------------------------

        title_element = item.select_one(
            '[data-testid="title"]'
        )

        if title_element:
            movie_name = clean_text(
                title_element.get_text()
            )
        else:
            movie_name = ""

        # ----------------------------------------------------
        # Movie year and rating
        # ----------------------------------------------------

        metadata = item.select(
            "span.cli-title-metadata-item"
        )

        movie_year = ""

        if metadata:
            movie_year = clean_text(
                metadata[0].get_text()
            )

        # ----------------------------------------------------
        # Movie rating
        # ----------------------------------------------------

        rating_element = item.select_one(
            '[data-testid="ratingGroup--imdb-rating"]'
        )

        movie_rating = ""

        if rating_element:
            rating_text = clean_text(
                rating_element.get_text()
            )

            rating_match = re.search(
                r"\d+\.\d+",
                rating_text
            )

            if rating_match:
                movie_rating = rating_match.group()

        # ----------------------------------------------------
        # Movie URL
        # ----------------------------------------------------

        link_element = item.select_one("a")

        movie_url = ""

        if link_element:

            movie_url = link_element.get("href", "")

            if movie_url.startswith("/"):
                movie_url = (
                    "https://www.imdb.com" + movie_url
                )

        # ----------------------------------------------------
        # Store movie
        # ----------------------------------------------------

        if movie_name and movie_url:

            movies.append({
                "Movie_Name": movie_name,
                "Movie_Year": movie_year,
                "Movie_Rating": movie_rating,
                "Movie_URL": movie_url
            })

    except Exception as e:

        print("Error while reading movie:", e)


print(
    "Movies successfully collected:",
    len(movies)
)


# ============================================================
# 6. COLLECT REVIEWS
# ============================================================

all_reviews = []

print("\nStarting review collection...\n")


for movie_number, movie in enumerate(
    movies,
    start=1
):

    print(
        f"[{movie_number}/{len(movies)}] "
        f"{movie['Movie_Name']}"
    )

    try:

        # IMDb reviews URL
        reviews_url = (
            movie["Movie_URL"].rstrip("/")
            + "/reviews/"
        )

        review_response = requests.get(
            reviews_url,
            headers=headers,
            timeout=20
        )

        if review_response.status_code != 200:

            print(
                "  Could not access reviews. "
                "Status:",
                review_response.status_code
            )

            continue


        review_soup = BeautifulSoup(
            review_response.text,
            "html.parser"
        )


        # ----------------------------------------------------
        # Find review containers
        # ----------------------------------------------------

        review_items = review_soup.select(
            "article.user-review-item"
        )


        # Alternative selector
        if not review_items:

            review_items = review_soup.select(
                '[data-testid="review"]'
            )


        # Limit number of reviews
        review_items = review_items[
            :REVIEWS_PER_MOVIE
        ]


        for review in review_items:

            try:

                # ------------------------------------------------
                # Reviewer name
                # ------------------------------------------------

                reviewer_element = review.select_one(
                    '[data-testid="author-link"]'
                )

                if reviewer_element:

                    reviewer_name = clean_text(
                        reviewer_element.get_text()
                    )

                else:

                    reviewer_name = ""


                # ------------------------------------------------
                # Review title
                # ------------------------------------------------

                title_element = review.select_one(
                    '[data-testid="review-title"]'
                )

                if title_element:

                    review_title = clean_text(
                        title_element.get_text()
                    )

                else:

                    # Alternative selector
                    title_element = review.select_one(
                        "h3"
                    )

                    if title_element:
                        review_title = clean_text(
                            title_element.get_text()
                        )
                    else:
                        review_title = ""


                # ------------------------------------------------
                # Review text
                # ------------------------------------------------

                text_element = review.select_one(
                    '[data-testid="review-text"]'
                )

                if text_element:

                    review_text = clean_text(
                        text_element.get_text()
                    )

                else:

                    # Alternative selector
                    text_element = review.select_one(
                        ".text"
                    )

                    if text_element:

                        review_text = clean_text(
                            text_element.get_text()
                        )

                    else:

                        review_text = ""


                # ------------------------------------------------
                # Rating
                # ------------------------------------------------

                rating_element = review.select_one(
                    '[data-testid="review-rating"]'
                )

                if rating_element:

                    rating_text = clean_text(
                        rating_element.get_text()
                    )

                    rating_match = re.search(
                        r"\d+",
                        rating_text
                    )

                    if rating_match:
                        review_rating = rating_match.group()
                    else:
                        review_rating = ""

                else:

                    review_rating = ""


                # ------------------------------------------------
                # Review date
                # ------------------------------------------------

                date_element = review.select_one(
                    '[data-testid="review-date"]'
                )

                if date_element:

                    review_date = clean_text(
                        date_element.get_text()
                    )

                else:

                    time_element = review.select_one(
                        "time"
                    )

                    if time_element:

                        review_date = clean_text(
                            time_element.get_text()
                        )

                    else:

                        review_date = ""


                # ------------------------------------------------
                # Add review to dataset
                # ------------------------------------------------

                all_reviews.append({

                    "Movie_Name":
                        movie["Movie_Name"],

                    "Movie_Year":
                        movie["Movie_Year"],

                    "Reviewer_Name":
                        reviewer_name,

                    "Review_Title":
                        review_title,

                    "Review_Text":
                        review_text,

                    "Rating":
                        review_rating,

                    "Review_Date":
                        review_date,

                    "Movie_Rating":
                        movie["Movie_Rating"],

                    "Movie_URL":
                        movie["Movie_URL"]

                })


            except Exception as e:

                print(
                    "  Error reading review:",
                    e
                )


        # ----------------------------------------------------
        # Delay between requests
        # ----------------------------------------------------

        time.sleep(2)


    except requests.RequestException as e:

        print(
            "  Request error:",
            e
        )


# ============================================================
# 7. CREATE RAW DATASET
# ============================================================

print("\n" + "=" * 60)
print("CREATING RAW DATASET")
print("=" * 60)


raw_df = pd.DataFrame(all_reviews)


if raw_df.empty:

    print(
        "\nNo reviews were collected."
    )

    print(
        "IMDb may have changed its page structure "
        "or restricted the request."
    )

    raise SystemExit


# Save raw dataset
raw_df.to_csv(
    RAW_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nRaw dataset saved as:",
    RAW_FILE
)


print(
    "Number of raw records:",
    len(raw_df)
)


# ============================================================
# 8. DISPLAY RAW DATA
# ============================================================

print("\nFirst 5 raw records:")

print(
    raw_df.head()
)


# ============================================================
# 9. DATA CLEANING
# ============================================================

print("\n" + "=" * 60)
print("DATA CLEANING")
print("=" * 60)


clean_df = raw_df.copy()


# ------------------------------------------------------------
# Remove duplicate records
# ------------------------------------------------------------

before_duplicates = len(clean_df)

clean_df = clean_df.drop_duplicates()

after_duplicates = len(clean_df)

print(
    "\nDuplicate records removed:",
    before_duplicates - after_duplicates
)


# ------------------------------------------------------------
# Clean all text columns
# ------------------------------------------------------------

text_columns = [

    "Movie_Name",
    "Movie_Year",
    "Reviewer_Name",
    "Review_Title",
    "Review_Text",
    "Rating",
    "Review_Date",
    "Movie_Rating",
    "Movie_URL"

]


for column in text_columns:

    if column in clean_df.columns:

        clean_df[column] = (
            clean_df[column]
            .fillna("")
            .astype(str)
            .apply(clean_text)
        )


# ============================================================
# 10. REMOVE EMPTY REVIEWS
# ============================================================

before_empty = len(clean_df)

clean_df = clean_df[
    clean_df["Review_Text"].str.len() > 0
]

after_empty = len(clean_df)

print(
    "Empty reviews removed:",
    before_empty - after_empty
)


# ============================================================
# 11. REMOVE UNWANTED CHARACTERS
# ============================================================

clean_df["Review_Text"] = (
    clean_df["Review_Text"]
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)


clean_df["Review_Title"] = (
    clean_df["Review_Title"]
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)


# ============================================================
# 12. CONVERT RATING TO NUMERIC
# ============================================================

clean_df["Rating"] = pd.to_numeric(
    clean_df["Rating"],
    errors="coerce"
)


clean_df["Movie_Rating"] = pd.to_numeric(
    clean_df["Movie_Rating"],
    errors="coerce"
)


# ============================================================
# 13. CONVERT YEAR TO NUMERIC
# ============================================================

clean_df["Movie_Year"] = pd.to_numeric(
    clean_df["Movie_Year"],
    errors="coerce"
)


# ============================================================
# 14. CHECK INVALID RATINGS
# ============================================================

# IMDb user ratings normally range from 1 to 10.

clean_df.loc[
    ~clean_df["Rating"].between(1, 10),
    "Rating"
] = pd.NA


# ============================================================
# 15. CHECK MISSING VALUES
# ============================================================

print("\nMissing values:")

print(
    clean_df.isnull().sum()
)


# ============================================================
# 16. REMOVE DUPLICATES AGAIN
# ============================================================

clean_df = clean_df.drop_duplicates()


# ============================================================
# 17. RESET INDEX
# ============================================================

clean_df = clean_df.reset_index(
    drop=True
)


# ============================================================
# 18. SAVE CLEANED DATASET
# ============================================================

clean_df.to_csv(
    CLEAN_FILE,
    index=False,
    encoding="utf-8-sig"
)


print(
    "\nCleaned dataset saved as:",
    CLEAN_FILE
)


# ============================================================
# 19. DISPLAY FINAL DATASET
# ============================================================

print("\n" + "=" * 60)
print("FINAL CLEANED DATASET")
print("=" * 60)


print(
    clean_df.head(10)
)


# ============================================================
# 20. DATASET INFORMATION
# ============================================================

print("\nDataset shape:")

print(
    clean_df.shape
)


print("\nColumn names:")

print(
    clean_df.columns.tolist()
)


print("\nDataset information:")

print(
    clean_df.info()
)


# ============================================================
# 21. BASIC STATISTICS
# ============================================================

print("\nBasic statistics:")

print(
    clean_df.describe(
        include="all"
    )
)


# ============================================================
# 22. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

print(
    "Number of movies processed:",
    len(movies)
)

print(
    "Number of raw reviews:",
    len(raw_df)
)

print(
    "Number of cleaned reviews:",
    len(clean_df)
)

print(
    "Raw dataset:",
    RAW_FILE
)

print(
    "Cleaned dataset:",
    CLEAN_FILE
)

print(
    "\nPPS 1 DATA COLLECTION AND PREPROCESSING COMPLETED."
)

print("=" * 60)

IMDb MOVIE REVIEW DATA COLLECTION

Connecting to IMDb...
Status Code: 202

Unable to access the IMDb page.
Do not bypass any CAPTCHA or access restriction.


SystemExit: 

C:\Users\deeks\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
